#1.Load Dataset

In [55]:
import pandas as pd

df=pd.read_csv('/content/drive/MyDrive/movie recommendation system/tmdb_5000_movies.csv')

print(df.head(5))

      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...                en   
2  [{"id": 470, "nam

In [56]:
df = df[['id', 'title', 'genres', 'keywords', 'overview']]

print(df.shape)
print(df.head(5))


(4803, 5)
       id                                     title  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                            keywords  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...   
2  [{"id": 470, "name": "spy"}, {"id": 818, "name...   
3  [{"id": 849, "name": "dc comics"}, {"id": 853,...   
4  [{"id": 818, "name": "based on novel"},

###1. Handling missing values

In [57]:
df.dropna(inplace=True)
print(df.shape)

(4800, 5)


I dropped missing values instead of imputing them because these are text fields,filling them with placeholder text would introduce noise and hurt the quality of recommendations.

###2. Clean the Genres keywords Columns

In [58]:
import ast

def extract_names(t):
  try:
    item=ast.literal_eval(t)
    names=[i['name'] for i in item]
    return names
  except:
    return []

df['genres']=df['genres'].apply(extract_names)
df['keywords']=df['keywords'].apply(extract_names)



print(df.head(5))

       id                                     title  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                          genres  \
0  [Action, Adventure, Fantasy, Science Fiction]   
1                   [Adventure, Fantasy, Action]   
2                     [Action, Adventure, Crime]   
3               [Action, Crime, Drama, Thriller]   
4           [Action, Adventure, Science Fiction]   

                                            keywords  \
0  [culture clash, future, space war, space colon...   
1  [ocean, drug abuse, exotic island, east india ...   
2  [spy, based on novel, secret agent, sequel, mi...   
3  [dc comics, crime fighter, terrorist, secret i...   
4  [based on novel, mars, medallion, space travel...   

                   

In [59]:
df['genres']=df['genres'].apply(lambda x: ' '.join([i.replace(' ','')for i in x]))
df['keywords']=df['keywords'].apply(lambda x: ' '.join(x))



print(df.head(3))

       id                                     title  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   

                                    genres  \
0  Action Adventure Fantasy ScienceFiction   
1                 Adventure Fantasy Action   
2                   Action Adventure Crime   

                                            keywords  \
0  culture clash future space war space colony so...   
1  ocean drug abuse exotic island east india trad...   
2  spy based on novel secret agent sequel mi6 bri...   

                                            overview  
0  In the 22nd century, a paraplegic Marine is di...  
1  Captain Barbossa, long believed to be dead, ha...  
2  A cryptic message from Bond’s past sends him o...  


###3. Combine all columns int one

In [60]:
df['tags']=df['overview']+' '+df['genres']+' '+df['keywords']+' '+df['title']

print(df[['title','tags']].head(5))


                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                                tags  
0  In the 22nd century, a paraplegic Marine is di...  
1  Captain Barbossa, long believed to be dead, ha...  
2  A cryptic message from Bond’s past sends him o...  
3  Following the death of District Attorney Harve...  
4  John Carter is a war-weary, former military ca...  


In [61]:
print(df['tags'][0])

In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d Avatar


#2. Text Processing

###1.Convert every thing to lowercase

In [62]:
df['tags']=df['tags'].apply(lambda x:x.lower())
print(df['tags'][0])

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d avatar


###2.Fixing tenses

In [63]:
import nltk
nltk.download('punkt')
from nltk.stem.porter import PorterStemmer

ps=PorterStemmer()

df['tags'] = df['tags'].apply(lambda x: ' '.join([ps.stem(word) for word in x.split()]))

print(df['tags'][0])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultur clash futur space war space coloni societi space travel futurist romanc space alien tribe alien planet cgi marin soldier battl love affair anti war power relat mind and soul 3d avatar


#3.Vectorization with CountVectorizer

In [64]:
from sklearn.feature_extraction.text import CountVectorizer

cv=CountVectorizer(max_features=5000,stop_words='english')
vectors=cv.fit_transform(df['tags']).toarray()

print(vectors.shape)

(4800, 5000)


#4.Calculating similarity

In [65]:
from sklearn.metrics.pairwise import cosine_similarity

similarity=cosine_similarity(vectors)
print(similarity.shape)

print(similarity[0])

(4800, 4800)
[1.         0.0709997  0.05444655 ... 0.04742231 0.         0.        ]


#5.Recommendation Function

In [66]:
def recommend(movie):
  movie_index=df[df['title']==movie].index[0]

  distances=similarity[movie_index]

  movie_list=sorted(list(enumerate(distances)),
                    reverse=True,
                    key=lambda x:x[1])

  for i in movie_list[1:6]:
    print(df.iloc[i[0]].title)



#6.Testing & Saving

In [67]:
recommend("Avatar")

Aliens
Silent Running
Mission to Mars
Alien
Moonraker


In [68]:
recommend("Toy Story")

Toy Story 2
Toy Story 3
Small Soldiers
Child's Play 2
Child's Play


In [69]:
recommend("Titanic")

The Notebook
Veer-Zaara
Ghost Ship
Romance & Cigarettes
Poseidon


In [77]:
import pickle
pickle.dump(df, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

In [74]:
print(df.columns.tolist())

['id', 'title', 'genres', 'keywords', 'overview', 'tags']


In [76]:
print(df['id'].head())
print(df['id'].dtype)

0     19995
1       285
2    206647
3     49026
4     49529
Name: id, dtype: int64
int64
